In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 라이브러리

In [ ]:
# 데이터 전처리
import pandas as pd
import numpy as np
import gzip
import json
import copy

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# Torch 모델 구조
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# BERT Model
# Fine Tuning
from transformers import AutoModel, AutoTokenizer
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput


# sklearn 전처리, 데이터 분할, 성능 평가지표
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# Utils

In [ ]:
def extract_cls_embeddings(texts, tokenizer, model, batch_size=16):
    embeddings = []
    model.eval()
    device = model.device  # 모델이 할당된 디바이스 가져오기
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="Extracting [CLS] embeddings"):
            batch_texts = texts[i:i + batch_size]
            inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")
            inputs = {key: val.to(device) for key, val in inputs.items()}  # 모든 입력을 모델의 디바이스로 이동
            outputs = model(**inputs)
            cls_batch = outputs.last_hidden_state[:, 0, :]
            embeddings.append(cls_batch.cpu().numpy())  # GPU 메모리를 절약하기 위해 CPU로 이동 후 numpy 변환
    return np.vstack(embeddings)

In [ ]:
def extract_first_image_urls(images_list):
    # 비어있거나 잘못된 값 방지
    if not images_list or not isinstance(images_list, list) or len(images_list) == 0:
        return pd.Series([None, None, None])
    img = images_list[0]
    return pd.Series([
        img.get('small_image_url'),
        img.get('medium_image_url'),
        img.get('large_image_url')
    ])

# 데이터셋 로딩

In [5]:
# 데이터 로딩
df = pd.read_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Amazon_Clothing_Shoes_and_Jewerly_Img_Filtered_Top3.json')
print(df.shape)
df.head()

(67248, 27)


,rating,title_x,text,images_x,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,...,images_y,videos,store,categories,details,bought_together,subtitle,author,year,month
0,4,Pretty bit could be lighter steel color,Pretty colored earrings but color of stainless...,[{'small_image_url': 'https://images-na.ssl-im...,B08HQPWVCC,B08HQPWVCC,AFJBKPK5W56XWSNPQU2WW66ISWYQ,2021-04-19 00:44:13.135,0,False,...,[{'thumb': 'https://m.media-amazon.com/images/...,[],Udalyn,"[Clothing, Shoes & Jewelry, Women, Jewelry, Ea...",{'Package Dimensions': '3.58 x 2.52 x 0.98 inc...,NaN,NaN,NaN,2021,4
1,5,Nice earrings.,Great little earrings. Perfect size. Not too b...,[{'small_image_url': 'https://images-na.ssl-im...,B08JQ88186,B08JQ88186,AFJBKPK5W56XWSNPQU2WW66ISWYQ,2020-11-11 22:56:37.523,0,False,...,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZENI,"[Clothing, Shoes & Jewelry, Women, Jewelry, Ea...",{'Package Dimensions': '5.59 x 3.98 x 0.71 inc...,NaN,NaN,NaN,2020,11
2,5,Works great,Great bracelet key chain. Perfect for busy mom...,[{'small_image_url': 'https://images-na.ssl-im...,B07YXXR1DG,B07YXXR1DG,AFJBKPK5W56XWSNPQU2WW66ISWYQ,2019-11-15 03:58:18.428,0,False,...,[{'thumb': 'https://m.media-amazon.com/images/...,[],Konikit,"[Clothing, Shoes & Jewelry, Women, Accessories...",{'Package Dimensions': '6 x 4.2 x 0.5 inches; ...,NaN,NaN,NaN,2019,11
3,4,Looks just like picture,This is a nice patriotic tank. The medium fits...,[{'small_image_url': 'https://images-na.ssl-im...,B07RHSJQQN,B07RHSJQQN,AFJBKPK5W56XWSNPQU2WW66ISWYQ,2019-06-05 21:16:27.979,0,False,...,[{'thumb': 'https://m.media-amazon.com/images/...,[],YEXIPO,"[Clothing, Shoes & Jewelry, Novelty & More, Cl...",{'Product Dimensions': '4 x 3 x 1 inches; 6 Ou...,NaN,NaN,NaN,2019,6
4,5,Nice comfortable shirt,Great boys shirt. My grandson wears a medium i...,[{'small_image_url': 'https://images-na.ssl-im...,B07LFZRYC2,B07LFZRYC2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,2019-05-24 20:18:58.171,0,False,...,[{'thumb': 'https://m.media-amazon.com/images/...,[],Hurley,"[Clothing, Shoes & Jewelry, Boys, Clothing, To...",{'Package Dimensions': '8.54 x 6.65 x 3.78 inc...,NaN,NaN,NaN,2019,5


# Load Model

## RoBERTa

In [6]:
# load roberta
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModel.from_pretrained("roberta-base")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# roberta embedding 추출 후 concat
text_embeddings = extract_cls_embeddings(df['text'].tolist(), tokenizer, model, batch_size=16)
text_embeddings = pd.DataFrame({'roberta_emb': [e for e in text_embeddings]})
df = pd.concat([df, text_embeddings], axis=1)

Extracting [CLS] embeddings: 100%|██████████| 4203/4203 [1:47:02<00:00,  1.53s/it]


## 불필요한 Column 제거

In [8]:
# 필요한 column 선별
text_final = df[['user_id', 'asin', 'parent_asin', 'text', 'images_x', 'rating', 'title_y', 'roberta_emb']]

# 첫번째 이미지만 사용
urls = text_final['images_x'].apply(extract_first_image_urls)
urls.columns = ['small_image_url', 'medium_image_url', 'large_image_url']
text_final = pd.concat([text_final, urls], axis=1)

# image_x column drop
text_final = text_final.drop(columns=['images_x'])
text_final.head()

,user_id,asin,parent_asin,text,rating,title_y,roberta_emb,small_image_url,medium_image_url,large_image_url
0,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08HQPWVCC,B08HQPWVCC,Pretty colored earrings but color of stainless...,4,Udalyn 6pairs Stud Earrings Set for Women Opal...,"[-0.084172376, 0.16061515, -0.014070798, -0.13...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
1,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B08JQ88186,B08JQ88186,Great little earrings. Perfect size. Not too b...,5,ZENI 925 Sterling Silver Snowflake Earrings fo...,"[-0.0825007, 0.17542093, -0.054644376, -0.1382...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
2,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07YXXR1DG,B07YXXR1DG,Great bracelet key chain. Perfect for busy mom...,5,Bangle key ring Bracelet Keychain wristlet key...,"[-0.09092549, 0.16201146, -0.038774576, -0.116...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
3,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07RHSJQQN,B07RHSJQQN,This is a nice patriotic tank. The medium fits...,4,YEXIPO Womens American Flag Racerback Tank Top...,"[-0.12041848, 0.089214586, -0.0277174, -0.1245...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...
4,AFJBKPK5W56XWSNPQU2WW66ISWYQ,B07LFZRYC2,B07LFZRYC2,Great boys shirt. My grandson wears a medium i...,5,Hurley Boys' One and Ony Graphic T-Shirt,"[-0.13693954, 0.07035728, -0.06032884, -0.1145...",https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...,https://images-na.ssl-images-amazon.com/images...


In [ ]:
# 최종 Text Emb Dataset 저장
text_final.to_json('/content/drive/MyDrive/Dataset/Amazon_Dataset/Preprocessed/Clothing_Shoes_v1.json')